<a href="https://colab.research.google.com/github/itsPronay/ViT-tinys-ai_hub/blob/main/Colab%20Benchmark%20result.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import torch
import time
import numpy as np
import timm
import pandas as pd

In [63]:
def benchmark_model(model, input_tensor, runs, run_warmup=True, warmup_runs=10):

  model.eval()

  #warmup
  with torch.no_grad():
    if run_warmup:
      for _ in range(warmup_runs):
        _ = model(input_tensor)

  #sync before timing (run on gpu only)
  if input_tensor.device.type == 'cuda':
    torch.cuda.synchronize()

  latencies = []

  for _ in range(runs):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(input_tensor)
    if input_tensor.device.type == 'cuda':
        torch.cuda.synchronize()
    latencies.append((time.perf_counter() - start) * 1000)

  mean_latency_ms = float(np.mean(latencies))
  min_latency_ms = float(np.min(latencies))
  max_latency_ms = float(np.max(latencies))
  std = float(np.std(latencies))

  result = {
      'mean_latency_ms' : round(mean_latency_ms, 4),
      'min_latency_ms' : round(min_latency_ms, 4),
      'max_latency_ms' : round(max_latency_ms, 4),
      'std_latency_ms' : round(max_latency_ms, 4),
  }

  return result

In [64]:
models = [
    "vit_tiny_patch16_224",
    "mobilevitv2_100",
    "mobilevitv2_125",
    "tiny_vit_5m_224"
]

image_sizes = [
    224,
    448
]

In [65]:
devices_to_test = ['cpu']
if torch.cuda.is_available():
    devices_to_test.append('cuda')
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found – CPU only.")

GPU detected: Tesla T4


In [66]:

def main():

  all_results = []
  vit_models = ["vit_tiny_patch16_224", "tiny_vit_5m_224"]

  for device in devices_to_test:
    for size in image_sizes:
      for model_name in models:

        # these models do not support 448, so skipping it
        if size != 224 and (model_name == 'vit_tiny_patch16_224' or model_name=='tiny_vit_5m_224'):
          continue

        print(f"Device: {device}, Model: {model_name}, Image Size: {size}")

        model = timm.create_model(model_name, pretrained=False)
        model = model.to(device)

        input_tensor = torch.randn(1, 3, size, size).to(device)
        metrics = benchmark_model(model, input_tensor, runs=10)

        metrics.update({
            'device' : device,
            'image_size' : "1, 3, {0}, {0}".format(size),
            'model' : model_name,
        })

        all_results.append(metrics)

  from IPython.display import display
  df = pd.DataFrame(all_results)
  display(df)

if __name__ == '__main__':
    main()

Device: cpu, Model: vit_tiny_patch16_224, Image Size: 224
Device: cpu, Model: mobilevitv2_100, Image Size: 224
Device: cpu, Model: mobilevitv2_125, Image Size: 224
Device: cpu, Model: tiny_vit_5m_224, Image Size: 224
Device: cpu, Model: mobilevitv2_100, Image Size: 448
Device: cpu, Model: mobilevitv2_125, Image Size: 448
Device: cuda, Model: vit_tiny_patch16_224, Image Size: 224
Device: cuda, Model: mobilevitv2_100, Image Size: 224
Device: cuda, Model: mobilevitv2_125, Image Size: 224
Device: cuda, Model: tiny_vit_5m_224, Image Size: 224
Device: cuda, Model: mobilevitv2_100, Image Size: 448
Device: cuda, Model: mobilevitv2_125, Image Size: 448


,mean_latency_ms,min_latency_ms,max_latency_ms,std_latency_ms,device,image_size,model
0,36.6904,34.9155,40.6072,40.6072,cpu,"1, 3, 224, 224",vit_tiny_patch16_224
1,58.9234,55.7530,62.2765,62.2765,cpu,"1, 3, 224, 224",mobilevitv2_100
2,80.5805,78.2178,86.3820,86.3820,cpu,"1, 3, 224, 224",mobilevitv2_125
3,68.0465,64.3809,76.1034,76.1034,cpu,"1, 3, 224, 224",tiny_vit_5m_224
4,232.9667,195.7384,303.6156,303.6156,cpu,"1, 3, 448, 448",mobilevitv2_100
5,334.5377,314.6724,354.7217,354.7217,cpu,"1, 3, 448, 448",mobilevitv2_125
6,5.0084,4.8591,5.1724,5.1724,cuda,"1, 3, 224, 224",vit_tiny_patch16_224
7,9.3420,8.5658,13.0174,13.0174,cuda,"1, 3, 224, 224",mobilevitv2_100
8,8.4642,8.3872,8.5959,8.5959,cuda,"1, 3, 224, 224",mobilevitv2_125
9,8.6850,8.4644,8.9246,8.9246,cuda,"1, 3, 224, 224",tiny_vit_5m_224
